In [1]:
import geopandas as gp
import pandas as pd
import os
import numpy as np
import re
from collections import Counter

# Alabama 2024 Primary Runoff Election Results

In [2]:
path = './raw-from-source/2024 Primary Runoff Precinct Results/'

files = os.listdir(path)

In [3]:
county_list = []
for file in files: 
    temp = pd.read_excel(path + file)
    county_name = file.split("-")[-1][0:-4]
    if county_name == "StClair":
        county_name = "St. Clair"
    temp["Party"] = temp["Party"].str.strip()
    temp["Party"] = temp["Party"].fillna("")
    temp["pivot_col"] = temp["Contest Title"].str.strip()+"-:-"+temp["Candidate"].str.strip()
    temp["pivot_col"] = np.where(temp["Party"]=="",temp["pivot_col"],temp["pivot_col"]+"-:-"+temp["Party"].str.strip())
#     temp["pivot_col"] = temp["pivot_col"].map({"STATE SENATOR, DISTRICT 6-:-Kyle Richard - Garrison-:-LIB":"STATE SENATOR, DISTRICT 6-:-Kyle Richard-Garrison-:-LIB"}).fillna(temp["pivot_col"])

    temp.drop(["Contest Title", "Party", "Candidate"], axis = 1, inplace = True)
    rename_dict = {i:i+"-:-"+county_name for i in temp.columns if i != "pivot_col"}
    temp.rename(columns = rename_dict, inplace = True)
    temp_transpose = temp.set_index("pivot_col").T
    temp_transpose.reset_index(inplace = True, drop = False)
    temp_transpose["County"] = county_name
    #print(temp_transpose.head(10))
    #break
    print(temp_transpose.head())
    county_list.append(temp_transpose)

pivot_col                         index  \
0                       ABSENTEE-:-Clay   
1          BIBB GRAVES SCH/COMM_-:-Clay   
2               COURTHOUSE ANNEX-:-Clay   
3                FARMER'S MARKET-:-Clay   
4                    PROVISIONAL-:-Clay   

pivot_col  REGISTERED VOTERS - TOTAL-:-Registered Voters - Total  \
0                                                          0       
1                                                          0       
2                                                          0       
3                                                          0       
4                                                          0       

pivot_col  BALLOTS CAST - TOTAL-:-Ballots Cast - Total  \
0                                                   96   
1                                                   13   
2                                                   19   
3                                                  105   
4                                         

pivot_col                             index  \
0                        ABSENTEE-:-Russell   
1              AUSTIN SUMBRY PARK-:-Russell   
2          CENTRAL ACTIVITES BLDG-:-Russell   
3                   COTTONTON VFD-:-Russell   
4               CRAWFORD COMM CTR-:-Russell   

pivot_col  REGISTERED VOTERS - TOTAL-:-Registered Voters - Total  \
0                                                          0       
1                                                          0       
2                                                          0       
3                                                          0       
4                                                          0       

pivot_col  BALLOTS CAST - TOTAL-:-Ballots Cast - Total  \
0                                                   30   
1                                                  137   
2                                                  136   
3                                                   57   
4                 

In [4]:
comb = pd.concat(county_list)

In [5]:
list(comb.columns)

['index',
 'REGISTERED VOTERS - TOTAL-:-Registered Voters - Total',
 'BALLOTS CAST - TOTAL-:-Ballots Cast - Total',
 'BALLOTS CAST - DEMOCRAT-:-Ballots Cast - Alabama Democratic P-:-DEM',
 'BALLOTS CAST - REPUBLICAN-:-Ballots Cast - Alabama Republican P-:-REP',
 'BALLOTS CAST - NON-PARTISAN-:-Ballots Cast - Nonpartisan',
 'BALLOTS CAST - BLANK-:-Ballots Cast - Blank',
 'MEMBER, CLAY COUNTY COMMISSION, DISTRICT NO. 5-:-Terry Heflin Sr.-:-DEM',
 'MEMBER, CLAY COUNTY COMMISSION, DISTRICT NO. 5-:-Beverly Appleby Hill-:-DEM',
 'MEMBER, CLAY COUNTY COMMISSION, DISTRICT NO. 5-:-Over Votes-:-DEM',
 'MEMBER, CLAY COUNTY COMMISSION, DISTRICT NO. 5-:-Under Votes-:-DEM',
 'County',
 'MEMBER, FRANKLIN COUNTY COMMISSION, DISTRICT NO. 1-:-Curtis Baker-:-REP',
 'MEMBER, FRANKLIN COUNTY COMMISSION, DISTRICT NO. 1-:-Micheal Murray-:-REP',
 'MEMBER, FRANKLIN COUNTY COMMISSION, DISTRICT NO. 1-:-Over Votes-:-REP',
 'MEMBER, FRANKLIN COUNTY COMMISSION, DISTRICT NO. 1-:-Under Votes-:-REP',
 'MEMBER, FRANKLIN

In [6]:
[i for i in comb.columns if "REPRESENTATIVE" in i]

['UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Anthony Daniels-:-DEM',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Dick Brewbaker-:-REP',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Shomari Figures-:-DEM',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Caroleene Dobson-:-REP',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Over Votes-:-DEM',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Over Votes-:-REP',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Under Votes-:-DEM',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Under Votes-:-REP']

In [7]:
comb_list = [i for i in comb.columns if "REPRESENTATIVE" in i] 

In [8]:
comb_list = [i for i in comb_list if "Under Votes" not in i and "Over Votes" not in i]

In [9]:
comb_list

['UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Anthony Daniels-:-DEM',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Dick Brewbaker-:-REP',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Shomari Figures-:-DEM',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Caroleene Dobson-:-REP']

In [10]:
def get_race(race_string):
    race_string = race_string.title()
    race_string = race_string.replace("(Vote For 1)","")
    if "U.S. House" in race_string or 'Us House' in race_string or "United States Representative" in race_string:
        return "CON"
    elif "United States Senator" in race_string:
        return "USS"
    elif "State House" in race_string or "State Representative" in race_string:
        return "SL"
    elif "State Senator" in race_string:
        return "SU"
    elif "President" in race_string:
        return "PRE"
    elif "US Senate" in race_string or "Us Senate" in race_string:
        return "USS"
    elif "Public Service" in race_string:
        if "1" in race_string:
            return "PS1"
        elif "2" in race_string:
            return "PS2"
    elif "Attorney General" in race_string:
        return "ATG"
    elif "Auditor General" in race_string or "State Auditor" in race_string:
        return "AUD"
    elif "Treasurer" in race_string:
        return "TRE"
    elif "Superintendent" in race_string:
        return "SUP"
    elif "Secretary Of State" in race_string:
        return "SOS"
    elif "Lieutenant Governor" in race_string:
        return "LTG"
    elif "Governor" in race_string:
        return "GOV"
    elif "Commissioner Of Labor" in race_string:
        return "LAB"
    elif "Commissioner Of Agriculture" in race_string:
        return "AGR"
    elif "Commissioner Of Insurance" in race_string:
        return "INS"
    elif "Associate Justice Of The Supreme Court" in race_string:
        if "5" in race_string:
            return "AJ5"
        elif "6" in race_string:
            return "AJ6"
    elif "Constitution" in race_string:
        return "CNS"
    elif "Amendment" in race_string:
        if "One " in race_string:
            return "A01"
        elif "Two " in race_string:
            return "A02"
        elif "Three " in race_string:
            return "A03"
        elif "Four " in race_string:
            return "A04"
        elif "Five " in race_string:
            return "A05"
        elif "Six " in race_string:
            return "A06"
        elif "Seven " in race_string:
            return "A07"
        elif "Eight " in race_string:
            return "A08"
        elif "Nine " in race_string:
            return "A09"
        elif "Ten " in race_string:
            return "A10"
        else:
            print("No race for:", race_string)
            raise ValueError
    elif "Referendum" in race_string:
        if "1" in race_string or "A" in race_string:
            return "RFA"
        elif "2" in race_string or "B" in race_string:
            return "RFB"
        else:
            print("No race for:", race_string)
            raise ValueError
    else:
        print("No race for:", race_string)
        raise ValueError
        
def get_election_type(race_string):
    if "(runoff)" in race_string:
        return "R"
    else:
        return "G"
        
def get_party(race_string):
    if "REP" in race_string:
        return "R"
    elif "DEM" in race_string or "(Democrat)" in race_string:
        return "D"
    elif "LIB" in race_string or "(L)" in race_string:
        return "L"
    elif "NON" in race_string:
        return "O"
    elif "IND" in race_string:
        return "I"
    elif race_string[0:3]=="Yes":
        return "YES"
    elif race_string[0:2]=="No":
        return "NO"
    else:
        print("NO RACE", race_string)
        return ""
           
def get_name(name_string):
    if "Write-In" in name_string:
        return "WRI"
    if "AMENDMENT" not in name_string and "Referendum" not in name_string and "CONSTITUTION" not in name_string:
        #print(name_string)
        #name_string = name_string.split(" (")[0]
        name_string = name_string.replace("'","")
        likely_last = name_string.split(" ")[-1]
        proposed_last = likely_last[:3]
        if proposed_last in ['II', 'III', 'Jr', 'Jr.', 'Sr.', 'JR.', "JR", "IV"]:
            likely_last = name_string.split(" ")[-2]
            proposed_last = likely_last[:3]
        #print(proposed_last.upper())
        return proposed_last.upper()
    else:
        return name_string.split("-:-")[1].upper()
#     name_string = name_string.split("-:-")[1]
#     name_string = name_string.replace(" (I)","")
#     name_string = name_string.replace("'","")
#     likely_last = name_string.split(" ")[-1]
#     proposed_last = likely_last[:3]
#     if proposed_last in ['II', 'III', 'Jr', 'Jr.', 'Sr.', 'JR.', "JR", "IV"]:
#         likely_last = name_string.split(" ")[-2]
#         proposed_last = likely_last[:3]
#     return proposed_last.upper()

def get_district(race_string, fill_level):
    race_string = race_string.split("-:-")[0]
    race_string = race_string.replace(" (Vote For 1)","")
    if "UNITED STATES REPRESENTATIVE" in race_string:
        break_word = "REPRESENTATIVE, "
        temp = race_string.split(break_word)[1]
        temp = re.findall('\d*', temp)[0]
    elif "STATE REPRESENTATIVE" in race_string or "STATE SENATOR" in race_string:
        break_word = "DISTRICT "
        temp = race_string.split(break_word)[1]
    else:
        raise ValueError
    
    return temp.zfill(fill_level)

def column_rename_function(name_string):
    election_type = "R"
    year = "24"
    party = get_party(name_string.split("-:-")[-1])
    race = get_race(name_string)
    district = ""
    if race in ["CON", "SU"]:
        district = get_district(name_string, 2)
        year = ""
    elif race in ["SL"]:
        district = get_district(name_string, 3)
        year = ""
    
    name = get_name(name_string)
    if "CONSTITUTION" in name_string or "STATEWIDE AMENDMENT" in name_string:

        new_col_name = election_type + year + race + district +  name
    else:
        new_col_name = election_type + year + race + district + party + name
        print(election_type)
        print(year)
        print(race)
        print(district)
        print(name)
    if len(new_col_name) > 10:
        print(name_string)
        print(new_col_name)
    return new_col_name

# Make a dictionary that points to the new column names and checks for duplicates
race_columns = comb_list

race_updates_dict = {}
race_updates_reversed = {}
clean_dups = {}
new_names = []
for val in race_columns:
    new_name = column_rename_function(val)
    race_updates_dict[val] = new_name
    if new_name not in new_names:
        new_names.append(new_name)
        race_updates_reversed[new_name] = val
    else:
        print("Duplicate", new_name)
        print(race_updates_reversed[new_name])
        print(val)
        clean_dups[val] = race_updates_reversed[new_name]

R

CON
02
DAN
R

CON
02
BRE
R

CON
02
FIG
R

CON
02
DOB


In [11]:
race_updates_dict

{'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Anthony Daniels-:-DEM': 'RCON02DDAN',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Dick Brewbaker-:-REP': 'RCON02RBRE',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Shomari Figures-:-DEM': 'RCON02DFIG',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Caroleene Dobson-:-REP': 'RCON02RDOB'}

In [12]:
myKeys = list(race_updates_dict.values())
myKeys.sort()
#sorted_dict = {myKeys: race_updates_dict[i] for i in myKeys}


sorted_dict = dict(sorted(race_updates_dict.items(), key=lambda x:x[1]))
export_dict = {i:key for key, i in sorted_dict.items()}

In [13]:
sorted_dict

{'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Anthony Daniels-:-DEM': 'RCON02DDAN',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Shomari Figures-:-DEM': 'RCON02DFIG',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Dick Brewbaker-:-REP': 'RCON02RBRE',
 'UNITED STATES REPRESENTATIVE, 2ND CONGRESSIONAL DISTRICT-:-Caroleene Dobson-:-REP': 'RCON02RDOB'}

In [14]:
pd.DataFrame(export_dict.items()).to_csv("./field_names_prim_runoff.csv", index = False)

In [15]:
comb.rename(columns = sorted_dict, inplace = True)
comb = comb[["index"]+list(sorted_dict.values())]
comb = comb.fillna(0)

In [16]:
for i in list(sorted_dict.values()):
    comb[i] = comb[i].astype(int)

In [17]:
county = pd.read_csv("/Users/peterhorton/Documents/RDH/raw_data/census/PL_COUNTYFP_NAMES.csv")
al_cnty_fips_dict = dict((zip(county[county["STUSAB"]=="AL"]["NAME"],county[county["STUSAB"]=="AL"]["COUNTYFP"].astype(str).str.zfill(3))))

In [18]:
comb["Precinct"] = comb["index"].apply(lambda x: x.split("-:-")[0])

In [19]:
comb["County"] = comb["index"].apply(lambda x: x.split("-:-")[1])

comb["County_map"] = comb["County"] + " County"

In [20]:
comb["COUNTYFP"] = comb["County_map"].map(al_cnty_fips_dict).fillna(comb["County"])

In [21]:
comb["index"] = comb["index"].apply(lambda x: x.split("-:-")[1]+"-:-"+x.split("-:-")[0])

In [22]:
comb.rename(columns = {"index":"UNIQUE_ID"}, inplace = True)

In [23]:
comb = comb[["UNIQUE_ID","COUNTYFP","County","Precinct"]+list(sorted_dict.values())]

In [24]:
comb["COUNTYFP"].unique()

array(['027', '059', '101', '085', '013', '109', '015', '087', '037',
       '127', '049', '099', '093', '073', '011', '133', '129', '045',
       '035', '097', '029', '041', '039', '115', '103', '113', '005',
       '025'], dtype=object)

In [25]:
al_cnty_fips_dict

{'Autauga County': '001',
 'Baldwin County': '003',
 'Barbour County': '005',
 'Bibb County': '007',
 'Blount County': '009',
 'Bullock County': '011',
 'Butler County': '013',
 'Calhoun County': '015',
 'Chambers County': '017',
 'Cherokee County': '019',
 'Chilton County': '021',
 'Choctaw County': '023',
 'Clarke County': '025',
 'Clay County': '027',
 'Cleburne County': '029',
 'Coffee County': '031',
 'Colbert County': '033',
 'Conecuh County': '035',
 'Coosa County': '037',
 'Covington County': '039',
 'Crenshaw County': '041',
 'Cullman County': '043',
 'Dale County': '045',
 'Dallas County': '047',
 'DeKalb County': '049',
 'Elmore County': '051',
 'Escambia County': '053',
 'Etowah County': '055',
 'Fayette County': '057',
 'Franklin County': '059',
 'Geneva County': '061',
 'Greene County': '063',
 'Hale County': '065',
 'Henry County': '067',
 'Houston County': '069',
 'Jackson County': '071',
 'Jefferson County': '073',
 'Lamar County': '075',
 'Lauderdale County': '077',
 

In [26]:
comb["COUNTYFP"].unique()

array(['027', '059', '101', '085', '013', '109', '015', '087', '037',
       '127', '049', '099', '093', '073', '011', '133', '129', '045',
       '035', '097', '029', '041', '039', '115', '103', '113', '005',
       '025'], dtype=object)

In [27]:
comb["UNIQUE_ID"] = comb["COUNTYFP"] + "-:-" + comb["Precinct"]

In [28]:
comb.sort_values(["COUNTYFP","Precinct"], inplace = True)

In [29]:
comb.groupby("County").sum()

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_36988/2048133636.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  comb.groupby("County").sum()


pivot_col,RCON02DDAN,RCON02DFIG,RCON02RBRE,RCON02RDOB
County,,,,
Barbour,663,362,346,328
Bullock,1304,99,42,90
Butler,269,422,333,927
Calhoun,0,0,0,0
Clarke,215,306,36,97
Clay,0,0,0,0
Cleburne,0,0,0,0
Conecuh,297,688,186,580
Coosa,0,0,0,0


In [30]:
for i in list(sorted_dict.values()):
    print(i, sum(comb[i]))

RCON02DDAN 14006
RCON02DFIG 21962
RCON02RBRE 10471
RCON02RDOB 14705


Republican totals match those certified by the party, found here: https://www.sos.alabama.gov/sites/default/files/election-2024/2024PrimaryRunoffRepublicanCandidateResults.pdf


Democratic party certification (https://www.sos.alabama.gov/sites/default/files/election-2024/2024PrimaryRunoffDemocraticCandidateResults.pdf) does not include statewide totals, but results are close to matching the unofficial election night totals, where for Figures they match exactly, but for Daniels, 16 more votes in the precinct-level file, than the unofficial election-night results, for Figures 36 more votes

Daniels (Barbour 2, Macon 1, Mobile 2, Montgomery 11)

Figures (Bulter 1, Clarke 1, Macon 2, Mobile 18, Montgomery 14)

# Export Election Results

In [31]:
comb.to_csv("./al_prim_run_2024_prec_csv.csv", index = False)